# Проверка match_192 на 192 ответах

- Прогон нового правила на тех же 192 ответах с полными эталонами и pymorphy3.
- Генерация таблицы для вслепой разметки 100 ответов вне первой выборки.

In [1]:
!pip install -q pymorphy3 pymorphy3-dicts-ru openpyxl
!git clone -q https://github.com/EgorCOStylev/rubq-answer-form-check /kaggle/working/repo
!git clone -q --depth 1 https://github.com/vladislavneon/RuBQ.git /kaggle/working/RuBQ
import os
os.chdir('/kaggle/working/repo')
os.environ['RUBQ_JSON'] = '/kaggle/working/RuBQ/RuBQ_2.0/RuBQ_2.0_test.json'

fatal: destination path '/kaggle/working/repo' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/RuBQ' already exists and is not an empty directory.


## Часть 1. Прогон на тех же 192

In [2]:
import sys
sys.path.insert(0, 'src')

import json
import math
import random
from pathlib import Path

import pandas as pd
from match_192 import correct_v2
from sp_common import load_rubq

DATA = Path('data/main/generations_main.jsonl')
XLSX_200 = Path('results/manual_check_200.xlsx')
UIDS_100 = Path('data/subsets/manual_check_192_uids.json')
OUT_XLSX = 'manual_check_192_100.xlsx'
SEED = 20261118

rows = {}
with open(DATA, encoding='utf-8') as f:
    for line in f:
        r = json.loads(line)
        rows[r['uid']] = r

rubq = load_rubq()

# --- подвыборка 100 для слепой разметки ---
if UIDS_100.exists():
    with open(UIDS_100, encoding='utf-8') as f:
        uids_100 = json.load(f)['uids']
    print('Подвыборка 100: загружена из файла')
else:
    first_uids = set(pd.read_excel(XLSX_200, sheet_name='manual_check')['uid'])
    all_uids = [r['uid'] for r in rows.values()]
    remaining = sorted(set(all_uids) - first_uids)
    uids_100 = sorted(random.Random(SEED).sample(remaining, 100))
    print('Подвыборка 100: сгенерирована')
print(f'Вторая подвыборка: {len(uids_100)} uids, seed={SEED}')

# --- данные для прогона 192 ---
df = pd.read_excel(XLSX_200, sheet_name='manual_check')
df = df[df['my_correct'].notna() & (df['my_correct'] != '?')].copy()
df['my_correct'] = df['my_correct'].astype(int)
print(f'Строк для сравнения: {len(df)}')

Подвыборка 100: загружена из файла
Вторая подвыборка: 100 uids, seed=20261118
Строк для сравнения: 192


In [3]:
def refs_of(row):
    refs = [x.strip() for x in str(row['ru_names']).split('|') if x.strip()]
    at = str(row['answer_text']).strip()
    if at and at not in refs and at.lower() != 'none':
        refs.append(at)
    return refs

def q_kind(q):
    t = q.get('answer_type', 'entity')
    return 'number' if t == 'number' else 'date' if t == 'date' else 'entity'

results = []
for _, row in df.iterrows():
    uid = row['uid']
    q = rubq[uid]
    kind = q_kind(q)
    refs = refs_of(row)
    ans = str(row['greedy'])
    v2_main = int(correct_v2(ans, refs, kind))
    v2_soft = int(correct_v2(ans, refs, kind, soft=True))
    results.append({
        'uid': uid,
        'kind': kind,
        'greedy': ans,
        'auto_correct': row['auto_correct'],
        'v2_main': v2_main,
        'v2_soft': v2_soft,
        'my_correct': row['my_correct'],
        'comment': row.get('comment', ''),
    })

res_df = pd.DataFrame(results)

In [4]:
def wilson(k, n):
    z = 1.96
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return round(100 * (c - h), 1), round(100 * (c + h), 1)

dis_main = res_df[res_df['v2_main'] != res_df['my_correct']]
dis_soft = res_df[res_df['v2_soft'] != res_df['my_correct']]

n = len(res_df)
lo, hi = wilson(len(dis_main), n)
los, his = wilson(len(dis_soft), n)

print(f'Основной вариант: {len(dis_main)}/{n} = {len(dis_main)/n:.1%} [{lo}; {hi}]')
print(f'Мягкий вариант:   {len(dis_soft)}/{n} = {len(dis_soft)/n:.1%} [{los}; {his}]')

print('\nОстаток основного:')
for _, r in dis_main.iterrows():
    print(f"  uid={r['uid']:>5} {r['kind']:<7} greedy={r['greedy'][:30]:<30} auto={r['auto_correct']} v2={r['v2_main']} my={r['my_correct']}")

# Сохранение результатов прогона 192
result_192 = {
    'n': n,
    'main': {'disagreements': len(dis_main), 'rate': round(len(dis_main)/n, 4), 'ci': wilson(len(dis_main), n)},
    'soft': {'disagreements': len(dis_soft), 'rate': round(len(dis_soft)/n, 4), 'ci': wilson(len(dis_soft), n)},
    'details': dis_main[['uid', 'kind', 'greedy', 'auto_correct', 'v2_main', 'my_correct']].to_dict('records')
}
with open('results/match_192_check.json', 'w', encoding='utf-8') as f:
    json.dump(result_192, f, ensure_ascii=False, indent=2)
print('Результаты сохранены: results/match_192_check.json')

Основной вариант: 23/192 = 12.0% [8.1; 17.3]
Мягкий вариант:   12/192 = 6.2% [3.6; 10.6]

Остаток основного:
  uid=  328 entity  greedy=Россия                         auto=0 v2=0 my=1
  uid=  588 entity  greedy=Вашингтон Кэпитолинс           auto=0 v2=0 my=1
  uid=  862 entity  greedy=Theseus                        auto=1 v2=0 my=1
  uid= 6090 entity  greedy=Сатира                         auto=1 v2=1 my=0
  uid= 6175 entity  greedy=Кремль                         auto=0 v2=0 my=1
  uid= 6343 entity  greedy=Гражданская война              auto=0 v2=0 my=1
  uid= 6366 entity  greedy=Листы бумаги                   auto=0 v2=0 my=1
  uid= 6472 entity  greedy=Ислам                          auto=0 v2=0 my=1
  uid= 6633 entity  greedy=IX век                         auto=0 v2=0 my=1
  uid= 6689 entity  greedy=1860                           auto=1 v2=0 my=1
  uid= 6740 entity  greedy=1976                           auto=0 v2=0 my=1
  uid= 6754 entity  greedy=2016                           auto=1 v

## Часть 2. Генерация таблицы для слепой разметки 100

In [5]:
import json
from pathlib import Path

UIDS_100_FILE = Path('data/subsets/manual_check_192_uids.json')
UIDS_100_FILE.parent.mkdir(parents=True, exist_ok=True)

record = {
    'seed': SEED,
    'n': len(uids_100),
    'source': str(DATA),
    'uids': uids_100,
}
with open(UIDS_100_FILE, 'w', encoding='utf-8') as f:
    json.dump(record, f, ensure_ascii=False, indent=2)
print(f'Подвыборка 100 сохранена: {UIDS_100_FILE}')

Подвыборка 100 сохранена: data/subsets/manual_check_192_uids.json


In [6]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font

COLS = [('uid', 7), ('kind', 8), ('question', 50), ('answer_text', 22),
        ('ru_names', 40), ('greedy', 25), ('auto_correct', 8),
        ('my_correct', 11), ('comment', 40)]

def ru_names(q, limit=12):
    out = []
    for a in q['answers']:
        out += [a.get('label') or a['value']] + a['wd_names']['ru'] + a.get('wp_names', [])
    return ' | '.join(list(dict.fromkeys(s for s in out if s))[:limit])

def refs_of_q(q):
    refs = ru_names(q).split(' | ')
    at = q.get('answer_text', '').strip()
    if at and at not in refs and at.lower() != 'none':
        refs.append(at)
    return refs

wb = Workbook()
ws = wb.active
ws.title = 'manual_check_192'
ws.append([c for c, _ in COLS])

for u in uids_100:
    q = rubq[u]
    r = rows[u]
    kind = q_kind(q)
    refs = refs_of_q(q)
    auto = int(correct_v2(r['greedy'], refs, kind))
    ws.append([u, kind, q['question_text'], q.get('answer_text', ''),
               ru_names(q), r['greedy'], auto, None, None])

for i, (name, width) in enumerate(COLS, start=1):
    letter = ws.cell(row=1, column=i).column_letter
    ws.column_dimensions[letter].width = width
    ws.cell(row=1, column=i).font = Font(bold=True)
    if name == 'auto_correct':
        ws.column_dimensions[letter].hidden = True

for row in ws.iter_rows(min_row=2):
    for c in row:
        c.alignment = Alignment(wrap_text=True, vertical='top')

ws.freeze_panes = 'A2'
wb.save(OUT_XLSX)
print(f'Сохранено: {OUT_XLSX}')

Сохранено: manual_check_192_100.xlsx


In [7]:
!cd /kaggle/working && tar czf match_192_out.tgz repo/results/main/*.json repo/manual_check_192_100.xlsx